# Energizados — Detección de Pérdidas No Técnicas

> Notebook paso a paso que recorre el pipeline completo del framework:
> carga de datos, partición temporal, modelos simples (*baseline*),
> ingeniería de variables, selección con Boruta, y entrenamiento de modelos supervisados.


### Install energizados package

In [ ]:
# !pip install energizados

### Configuración (paths y archivos)


In [ ]:
# ============================================================
# CONFIGURACIÓN — paths, archivos y parámetros del dataset
# ============================================================
# Toda la configuración está acá. Para apuntar la notebook a otro
# proyecto o dataset, modificá estos valores (no hace falta tocar el
# resto de las celdas).

from pathlib import Path

# --- Proyecto y datos ---
# ⚠️ Cambiá esta ruta al path de tu proyecto.
PROJECT_PATH = Path('/home/vvv/Develop/bid/energizados/.proyects/celesc')
PROC_PATH = PROJECT_PATH / 'data/processed/v5'
DATASET_FILE = PROC_PATH / 'dataset_celesc_train.parquet'

# --- Partición temporal (split train / val / test por fecha) ---
# train: fecha < TRAIN_UNTIL | val: [TRAIN_UNTIL, VAL_UNTIL) | test: fecha >= VAL_UNTIL
TRAIN_UNTIL = '2025-09-01'
VAL_UNTIL = '2026-01-01'

# --- Nombres de columnas ---
# La notebook usa nombres 'canónicos' internamente. Si tu dataset usa otros
# nombres, mapealos acá (clave = nombre en el dataset, valor = nombre canónico).
# Celesc: periodo→fecha_inspeccion, material→material_instalacion, geo_region→zona.
COL_ALIASES = {
    'periodo': 'fecha_inspeccion',
    'material': 'material_instalacion',
    'geo_region': 'zona',
}

# Columna de fecha usada para el split temporal (nombre canónico).
DATE_COL = 'fecha_inspeccion'

# Variables categóricas y columnas a rellenar con 'sin_dato'.
CATEGORICAL_COLS = ['zona', 'actividad', 'material_instalacion', 'tipo_tarifa', 'nivel_tension']
COLS_FILLNA_SINDATO = ['zona', 'actividad', 'tipo_tarifa', 'nivel_tension']

# --- Entorno (detección automática Colab vs local) ---
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'PROJECT_PATH : {PROJECT_PATH}')
print(f'DATASET_FILE : {DATASET_FILE}')
print(f'Split        : train < {TRAIN_UNTIL} | val < {VAL_UNTIL} | test >= {VAL_UNTIL}')
print(f'IN_COLAB     : {IN_COLAB}')

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt

import tsfel
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Add src to path to import energizados modules
# Nota: En el repo local es '../src', en el repo clonado para Colab es 'src'
if IN_COLAB:
    module_path = os.path.abspath(os.path.join('src'))
else:
    module_path = os.path.abspath(os.path.join('../src'))
    
if module_path not in sys.path:
    sys.path.append(module_path)
    
print(f"Module path agregado: {module_path}")

In [ ]:
from energizados.preprocessing.preprocessing import (
    fill_empty_values_str,
    fill_empty_values_cycle,
    TsfelVars,
    ExtraVars,
    ToDummy,
    TeEncoder,
    CardinalityReducer
)
from energizados.feature_selection import (
    feature_selection_by_constant,
    feature_selection_by_correlation,
    BorutaSelector
)
from energizados.modeling.simple_models import (
    ChangeTrendPercentajeIdentifierWide,
    ConstantConsumptionClassifierWide
)

In [ ]:
warnings.filterwarnings('ignore')
pd.options.display.float_format = '{:.2f}'.format
pd.set_option('display.max_columns', None)
np.set_printoptions(suppress=True)

In [ ]:
seed = 42
np.random.seed(seed)

### Etapa 1: Preprocesamiento de datos / Exploración / Entendimiento

Se requiere de esta etapa para tomar los datos en crudos y darles una estructura de datos significativa. Aquí es importante remarcar que un entendimiento y exploración de datos también es primordial ya que en este paso también se puede decidir qué otras variables pueden usarse en el proceso de detección de pérdidas no técnicas.

En nuestras pruebas, Energizados se evaluó en dos conjuntos de datos provistos por dos empresas de distribución de energía. Se observó que además de las series de consumos mensuales, incorporar variables que describen a los usuarios también ayudan en la detección de usuarios fraudulentos.

Entre otras cosas que se pueden observar en esta etapa es la proporción de usuarios fraudulentos y no fraudulentos. En este tipo de problemas es común tener clases desbalanceadas: por lo general la proporción de usuarios fraudulentos **no supera el 10%**.


# Paso 1 - Leer datos
***

**Nota:** El dataset se define en `DATASET_FILE` (bloque de configuración del principio). Las columnas se renombran a nombres canónicos mediante `COL_ALIASES` y se agrega una columna `index` con el id de fila (la usa el paso de *feature engineering*).

Descripción de las columnas:

| Variable  | Descripción | Tipo de dato | Cardinalidad |
| :--- | :--- | :--- | :--- |
| Consumo de energía mensual | Indica el comportamiento de consumo a nivel mensual de los usuarios.  Se consideran los últimos 12 consumos.| Numérica | - |
| Actividad | Indica a qué actividad económica se dedica el usuario| Categoría | 284 |
| Tipo de Tarifa | Tarifa que tipo de tarifa se le cobra al usuario| Categoría | 47 |
| Tensión | Tensión instalada al usuario.| Categoría | 18 |
| Material instalacion | Indica tipo de material del medidor instalado| Categoría | 39 |
| Zona | Indica la ubicación geográfica a la que pertenece el usuario | Categoría | 38 |
| Target | Indica si hubo un comportamiento fraudulento o no | Numérica | 0 - 1 |
| Fecha inspección | Indica la fecha en que se inspeccionó al usuario| Fecha | - |

### Dataset

Las estadísticas exactas (registros, columnas y % de fraudulentos) dependen del dataset configurado en `DATASET_FILE`.


In [ ]:
# Cargar dataset configurado en DATASET_FILE.
# Se renombran las columnas a nombres canónicos (COL_ALIASES) y se agrega una
# columna 'index' con el id de fila (la usa el paso de feature engineering).
df = pd.read_parquet(DATASET_FILE)
df = df.rename(columns=COL_ALIASES).reset_index()

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.dtypes

In [ ]:
print("Proporcion de clase : ", 100*df.target.mean())

### Partición de datos

Previo a la construcción de modelos, se particiona el conjunto de datos en entrenamiento, validación y test utilizando un **split temporal** basado en la columna `fecha_inspeccion` (definida vía `DATE_COL`). Los cortes `TRAIN_UNTIL` y `VAL_UNTIL` se configuran al principio de la notebook. Esta estrategia respeta el orden cronológico de los datos, evitando la fuga de información futura hacia el entrenamiento.


# Paso 2 - Particionar datos
***

In [ ]:
# Partición temporal usando DATE_COL y los cortes definidos en la configuración.
df_train = df[df[DATE_COL] < TRAIN_UNTIL].copy()
df_val = df[(df[DATE_COL] >= TRAIN_UNTIL) & (df[DATE_COL] < VAL_UNTIL)].copy()
df_test = df[df[DATE_COL] >= VAL_UNTIL].copy()

In [ ]:
print(df_train.shape)
print(df_val.shape)
print(df_test.shape)

In [ ]:
print("Proporcion de clase train : ", 100*df_train.target.mean())
print("Proporcion de clase validacion : ", 100*df_val.target.mean())
print("Proporcion de clase test : ", 100*df_test.target.mean())

### Etapa 2: Construcción de Modelos

En esta etapa primeramente se evaluaron modelos simples o modelos *baseline* para luego desarrollar modelos más complejos.


# Paso 3 - Procesamiento de datos y construcción de modelos
***

In [ ]:
df_train.isnull().sum()

**<ins>Observación :</ins>** 

>En el conjunto de datos existen valores faltantes en las variables de consumo que son de tipos numericas y en las variables categóricas como "zona", "actividad", "tipo_tarifa" y "nivel_tension".

El tratamiento de valores faltantes fue abordado de la siguiente forma : 

- <ins>variables de consumo :</ins> se usaron los metodos ffill y bfill para propagar la observación válida hacia adelante o hacia atras.
- <ins>variables categoricas :</ins>  estas se rellenaron con una nueva categoria denominada "sin_dato".

**Nota:** En el framework actualizado, las funciones se llaman `fill_empty_values_cycle` y `fill_empty_values_str`.

In [ ]:
# Relleno de valores faltantes en serie de consumo.
df_train = fill_empty_values_cycle(df_train, 12)

# Relleno de valores faltantes en variables categóricas (definidas en config).
cols_fillna_sindatos = COLS_FILLNA_SINDATO
df_train = fill_empty_values_str(df_train, cols_fillna_sindatos, 'sin_dato')

In [ ]:
df_train.head()

#### Modelos Simples

Modelos que a través de reglas analíticas pueden detectar un comportamiento anómalo en el consumo de energía de los usuarios. Estas reglas en general se derivan luego de hacer un análisis exploratorio de los datos y también de conocimientos de expertos.

Se implementan dos reglas principales:
- **Cambio o disminución en el consumo de energía**: evalúa si un usuario disminuyó dramáticamente su consumo actual con respecto a períodos anteriores.
- **Consumo constante**: evalúa si el consumo fue constante por un período largo de tiempo.


## Modelos Simples

### Regla : Cambio o disminución en el consumo de energía

>La hipotesis detras de esta regla es que si se existen decrementos bruscos de consumos entonces es un posible comportamiento anomalo.

Configuración : 
- last_base_value : indica la cantidad de periodos anteriores para comparar.
- last_eval_value : indica la cantidad de consumos a ser evaluados.
- threshold : indica la proporción de consumo.

In [ ]:
variables_consumo = [x for x in df.columns if '_anterior' in x]
last_base_value,last_eval_value,threshold = 3,1,60
trend_perc_model = ChangeTrendPercentajeIdentifierWide(last_base_value,last_eval_value,threshold)
pred = trend_perc_model.predict(df_test[variables_consumo])

In [ ]:
# Existen un 10% de usuarios en test que cumplieron con la regla.
100*pred.is_fraud_trend_perc.value_counts(normalize=True)

In [ ]:
# usuario ejemplo que cumplio con la regla
usr = df_test.index[0]  # Ajustado para el nuevo dataset
df_test.loc[usr]

In [ ]:
plt.figure(figsize=(15,4))
y = df_test[variables_consumo].loc[usr].values
x = range(len(y))
plt.plot(x,y)
plt.scatter(x,y, color='red')
plt.ylim(0.0)
plt.grid(True)
plt.title("usr:" + str(usr)+" Cambio Trend último mes ")
plt.show()

### Regla : Consumos constante

> La hipotesis de esta regla es que si existen consumos constantes por periodos largos, entonces es un posible comportamiento anomalo.
- min_count_constante : indica la cantidad minima de periodos donde los consumo son constantes.


In [ ]:
min_count_constante =7
const_model = ConstantConsumptionClassifierWide(min_count_constante)
y_test_pred = const_model.predict(df_test[variables_consumo])

In [ ]:
# Existen aprox un 3% de usuarios en test que cumplieron con la regla.
100*y_test_pred.value_counts(normalize=True)

In [ ]:
# usuario ejemplo que cumplió con la regla
usr = df_test.index[1]  # Ajustado para el nuevo dataset
df_test.loc[usr]

In [ ]:
plt.figure(figsize=(15,4))
y = df_test[variables_consumo].loc[usr].values
x = range(len(y))
plt.plot(x,y)
plt.scatter(x,y, color='red')
plt.ylim(0.0)
plt.grid(True)
plt.title("usr:" + str(usr)+" Consumo constante ")
plt.show()

### Modelos Supervisados

En lo que respecta a la construcción de los modelos supervisados, se siguieron los siguientes pasos:

1. Construcción de variables (*feature engineering*)
2. Selección de variables (*feature selection*)
3. Manejo de datos desbalanceados
4. Optimización de hiperparámetros
5. Entrenamiento final con los mejores hiperparámetros


## Modelos Supervisados

#### Ingeniería de variables

La ingeniería de variables es el proceso de extracción y selección de las variables más importantes de los datos. Esta etapa puede dividirse en dos subtareas:

**Extracción de variables:** Los consumos mensuales por sí solos carecen de características estadísticas que reflejen adecuadamente los patrones subyacentes. Por lo tanto se crearon nuevas variables derivadas de los consumos, que pueden clasificarse en tres tipos:

- **Estadísticas**: máximo, promedio, mínimo, mediana.
- **Espectrales**: distancia de la señal, pendiente, varianza, etc.
- **Temporales**: autocorrelación, entropía, centroides, etc.

Para las variables categóricas que caracterizan a los usuarios (tipo de tarifa, actividad económica, etc.) se aplicaron transformaciones clásicas: variables *dummy*, reducción de cardinalidad y *target encoding*.


### Ingenieria de variables 

> El objetivo principal es derivar variables de las serie de consumo mensual. 

**Ejemplo :** 

    1. min, maximo, pendientes.
    2. variables estadisticas,temporaales y expectrales.
    
**Paquete :**

- [TSFEL](https://tsfel.readthedocs.io/en/latest/)
- [Ejemplo de uso](https://github.com/fraunhoferportugal/tsfel/blob/master/notebooks/TSFEL_SMARTWATCH_HAR_Example.ipynb)
- Otro paquete --> [TSFRESH](https://tsfresh.readthedocs.io/en/latest/)

En el siguiente ejemplo vemos una serie de consumo, luego con el paquete TSFEL, vamos a extrar variables estadisticas que luego lo podemos usar como variables predictoras en un modelo de supervisado.

In [ ]:
serie_consumo_anteriores = [153.0,  125.0,  117.0,  120.0,  128.0,  80.0,  105.0,  123.0,  101.0,  111.0,  99.0,  96.0]
plt.figure(figsize=(10,5))
plt.plot(serie_consumo_anteriores)
plt.xticks(range(12));

In [ ]:
cfg = tsfel.get_features_by_domain("statistical")
df_result = tsfel.time_series_features_extractor(cfg, serie_consumo_anteriores,n_jobs=-1)

In [ ]:
# Como resultados tenemos una diversidad de variables estadisticas como : 0_Max	0_Mean 0_Standard deviation	0_Variance, etc.
df_result.shape

In [ ]:
df_result[['0_Skewness','0_Kurtosis', '0_Standard deviation','0_Interquartile range', '0_Kurtosis', '0_Max', '0_Mean','0_Mean absolute deviation']]

#### Selección de variables

La selección de variables es el proceso de identificar un subconjunto representativo de variables de un grupo más grande. En el desarrollo de los modelos ejecutamos los siguientes pasos:

1. Eliminación de variables constantes o con muy baja variabilidad.
2. Eliminación de variables altamente correlacionadas entre sí.
3. Selección del grupo de variables relevantes mediante **Boruta**.


### Selección de variables 

**<ins>Nota:</ins>** En este ejemplo el proceso puede tardar más de 5 minutos! --> puede levantar las variables seleccionadas ya calculadas

> El objetivo es seleccionar las mejores variables para entrenar los modelos.

**Metodos y Paquete :**

- [Boruta](https://pypi.org/project/Boruta/)
- [Ejemplo de uso boruta](https://towardsdatascience.com/feature-selection-with-boruta-in-python-676e3877e596)
- [Mutual Information](https://towardsdatascience.com/select-features-for-machine-learning-model-with-mutual-information-534fe387d5c8)

Este paso lo realizamos luego de extraer las nuevas variables derivadas de las series de consumo. 

In [ ]:
# NOTE: Esta celda puede tardar varios minutos (TSFEL sobre 12.000 filas).
# Descomentar para ejecutar la selección de variables.
# Este paso lo vamos hacer con una muestra del conjunto de datos
variables_consumo = [x for x in df.columns if '_anterior' in x]
df_consumos = df_train[['index']+variables_consumo].head(12000)

# Construimos el pipeline de ingenieria de variables.
# TsfelVars --> Encapsula todas las funcionalidades del paquete TSFEL.
# ExtraVars --> Modulo que agrega variables extras, como cantidad de ceros seguidos en la serie de consumo y en distintas ventanas de tiempo.

pipe_feature_engeniering_consumo = Pipeline(
    [
        ("tsfel vars", TsfelVars(features_names_path=None, num_periodos=12)),
        ("add vars3",  ExtraVars(num_periodos=3)),
        ("add vars6",  ExtraVars(num_periodos=6)),
        ("add vars12", ExtraVars(num_periodos=12)),
    ]
)

df_features = pipe_feature_engeniering_consumo.fit_transform(df_consumos)

>Luego de crear nuevas variables vamos a aplicar los pasos para las seleccion de las variables mas importantes.

- Eliminamos varibles constantes
- Eliminamos las que esta altamente correlacionadas
- Seleccionamos con el metod boruta

In [ ]:
cols_for_feature_sel = [x for x in df_features.columns if x not in ['index'] + variables_consumo]
y_train = df_train.loc[df_features['index']].target

In [ ]:
%%time
select_by_constant = feature_selection_by_constant(df_features, y_train, cols_for_feature_sel, th=0.99)
print(f" # variables No constantes {len(select_by_constant)}")

select_by_corr = feature_selection_by_correlation(df_features, y_train, select_by_constant,method='pearson', th=0.95)
print(f" # variables No correlacionadas {len(select_by_corr)}")

boruta_selector = BorutaSelector(max_iter=5)
boruta_selector.fit(df_features[select_by_constant], y_train)
select_by_boruta = boruta_selector.get_selected_features()
print(f" # variables seleccionadas por Boruta : {len(select_by_boruta)}")

# Para la demo, select_by_boruta queda vacío (sin feature engineering de consumo extra)
# select_by_boruta = []  # Reemplazar por cols_for_feature_sel si se ejecutó el pipeline de arriba

In [ ]:
# Levantar variables ya seleccionadas previamente
# select_by_boruta = pd.read_csv('../data/preprocesados/features.csv')['features'].tolist()

In [ ]:
len(select_by_boruta)

### Tratamiento de las variables categoricas

> Las variables categóricas son un desafío para los algoritmos de Machine Learning. Dado que la mayoría de ellos aceptan solo valores numéricos como entradas, necesitamos transformar las categorías en números para usarlos en el modelo.

In [ ]:
# Variables categóricas (definidas en la configuración inicial).
variables_categoricas = CATEGORICAL_COLS

In [ ]:
df_train[variables_categoricas].head()

El tratamieno de cada variable es el siguiente : 

- __actividad__:

*Reducción de cardinalidad y dummy:* 

> Variables categóricas a las que se le redujo la cardinalidad (Esta reducción se logra, por ejemplo, agrupando valores escasos que no tienen una presencia importante en el set de datos) y luego se les aplicó One-Hot-Encoding.


- __tipo_tarifa__:

*Reducción de cardinalidad y target encoding:*

>Variables categóricas a las que se le redujo la cardinalidad y luego se las reemplazó por una medida del efecto que podrían tener en el objetivo.

- __zona y nivel_tension__:

*Variables encodeadas:*

>Variables categóricas a las que se les ha aplicado OrdinalEncoder.

- __material_instalacion__:

*Target encoding:*

>Variables categóricas a las que se le redujo la cardinalidad y luego se las reemplazó por una medida del efecto que podrían tener en el objetivo.

Nota : [Target-encoding](https://towardsdatascience.com/dealing-with-categorical-variables-by-using-target-encoder-a0f1733a4c69) 

_Finalmente el pipeline de preprocesamiento para las variables categoricas quedo configurado como se muestra a continuacion:_

```python

pipe_actividad = Pipeline([
            ('cardinality_reducer', CardinalityReducer(threshold=0.001)),
            ('a_dummy',ToDummy(['actividad']))
        ])


pipe_tarifa = Pipeline([
            ('cardinality_reducer', CardinalityReducer(threshold=0.001)),
            ('tarifa_te',TeEncoder(['tipo_tarifa'],w=20))
        ])

vars_enc = ['zona','nivel_tension']
t_features = [
    ('var_encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), vars_enc),
    ('material_isntalacion_te', TeEncoder(['material_instalacion'],w=10), ['material_instalacion']),
    ('actividad_cr_dummy', pipe_actividad, ['actividad']),
    ('tarifa_cr_te', pipe_tarifa, ['tipo_tarifa']),
    ]

preprocessor = ColumnTransformer(transformers= t_features,remainder='passthrough')

```

#### Entrenamiento de modelos

Como ocurre en la mayoría de los conjuntos de datos de detección de pérdidas no técnicas, los datos están **desbalanceados**. Para abordar este problema se utilizaron dos estrategias principales:

- **Sobremuestreo (oversampling)**: generar nuevas muestras de la clase infrarrepresentada.
- **Submuestreo (undersampling)**: eliminar ejemplos de la clase mayoritaria.

Se optó por *undersampling* ya que obtuvo mejores resultados en la fase de desarrollo.

Para la optimización de hiperparámetros se usó **Random Search**, que muestrea valores posibles y selecciona los que generan mejores métricas.


### Entrenamiento y evaluación de modelos supervisados

#### Procesamos los dataset de entrenamiento, validacion y test.

**<ins>Nota:</ins>** En este ejemplo este proceso puede tardar! --> puede levantar los dataset procesados

In [ ]:
y_train = df_train.target.copy()
df_train = df_train.drop(columns=['target'])

y_val = df_val.target.copy()
df_val = df_val.drop(columns=['target'])

y_test = df_test.target.copy()
df_test = df_test.drop(columns=['target'])

In [ ]:
# Realizamos los pasos de limpieza en los conjuntos de validacion y test.
df_val = fill_empty_values_cycle(df_val, 12)
df_val = fill_empty_values_str(df_val, cols_fillna_sindatos, 'sin_dato')

df_test = fill_empty_values_cycle(df_test, 12)
df_test = fill_empty_values_str(df_test, cols_fillna_sindatos, 'sin_dato')

In [ ]:
%%time
# Calculamos las variables derivadas de las series de consumo en los 3 conjuntos de datos.
df_train = pipe_feature_engeniering_consumo.fit_transform(df_train)
df_val = pipe_feature_engeniering_consumo.transform(df_val)
df_test = pipe_feature_engeniering_consumo.transform(df_test)

**<ins>Levantar datasets procesados :</ins>** 

In [ ]:
#Levantar previamente calculadas
# df_train = pd.read_parquet('../data/preprocesados/df_train_p.parquet')
# df_val = pd.read_parquet('../data/preprocesados/df_val_p.parquet')
# df_test = pd.read_parquet('../data/preprocesados/df_test_p.parquet')

In [ ]:
# Definimos las variables finales para el entrenamiento de los modelos.
feauture_selected = select_by_boruta
cols_for_model = variables_categoricas+variables_consumo+feauture_selected

In [ ]:
# Definimos el metodo de balanceo de clases con su correspondiente umbral y el pipeline de pre-procesamiento de variables categoricas.
param_imb_method = 'under'
sam_th = 0.2
periodo = 12

In [ ]:
resultado_final = {} # para guardar todas las metricas obtenidas

In [ ]:
def get_preprocesor(preprocesor):
    """Build a sklearn ColumnTransformer based on a preprocessor number.

    Args:
        preprocesor: Integer selecting the preprocessing configuration.
            Currently only 4 is supported.

    Returns:
        sklearn.compose.ColumnTransformer: Configured preprocessor.
    """
    if preprocesor == 4:
        pipe_actividad = Pipeline(
            [
                ("cardinality_reducer", CardinalityReducer(threshold=0.001)),
                ("a_dummy", ToDummy(["actividad"])),
            ]
        )

        pipe_tarifa = Pipeline(
            [
                ("cardinality_reducer", CardinalityReducer(threshold=0.001)),
                ("tarifa_te", TeEncoder(["tipo_tarifa"], w=20)),
            ]
        )

        vars_enc = ["zona", "nivel_tension"]
        t_features = [
            (
                "var_encoder",
                OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
                vars_enc,
            ),
            (
                "material_isntalacion_te",
                TeEncoder(["material_instalacion"], w=10),
                ["material_instalacion"],
            ),
            ("actividad_cr_dummy", pipe_actividad, ["actividad"]),
            ("tarifa_cr_te", pipe_tarifa, ["tipo_tarifa"]),
        ]

        preprocessor = ColumnTransformer(transformers=t_features, remainder="passthrough")

    return preprocessor

#### Modelos utilizados

**Light Gradient Boosting Machine (LightGBM)**

LGBM es un modelo de *gradient boosting* basado en árboles de decisión. Es un método de ensamble que construye modelos secuenciales: cada nuevo modelo se enfoca en predecir correctamente los casos donde el anterior tuvo desempeño deficiente.

Características destacadas: velocidad de entrenamiento, manejo eficiente de grandes volúmenes de datos con poca memoria, y soporte nativo para valores faltantes.

**Red Neuronal Multicapa**

Es una red *feedforward* donde todas las señales van en una misma dirección. Las entradas son las variables preprocesadas y la capa de salida devuelve la probabilidad de que un usuario esté cometiendo fraude.

**Red LSTM + Multicapa**

La LSTM (*Long Short-Term Memory*) es una red neuronal recurrente que tiene memoria interna, permitiéndole aprender patrones en secuencias temporales. En este proyecto, los consumos mensuales se tratan como una serie de tiempo. Se combinó una red LSTM (que procesa la secuencia de 12 consumos) con una red multicapa (que procesa las variables categóricas), concatenando ambas salidas para la clasificación final.

> **Nota:** Las redes neuronales no se ejecutan en esta notebook paso a paso, pero están disponibles a través del framework configurando `type: "neural_network"` o `type: "lstm"` en `config/train.yaml`.


#### LGBM

In [ ]:
%%time
# Importar librerías necesarias para el pipeline de LGBM
# IMPORTANTE: usar imblearn.pipeline.Pipeline (no sklearn) para soportar samplers
from imblearn.pipeline import Pipeline as ImbPipeline
from lightgbm import LGBMClassifier, early_stopping, log_evaluation
from sklearn.model_selection import RandomizedSearchCV
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from scipy.stats import randint as sp_randint, uniform as sp_uniform

# Obtener el pipeline de preprocesamiento para variables categóricas
preprocessor = get_preprocesor(4)

# Construir sampler para balanceo de clases
if param_imb_method == 'under':
    sampler = RandomUnderSampler(sampling_strategy=sam_th, random_state=40)
elif param_imb_method == 'over':
    sampler = RandomOverSampler(sampling_strategy=sam_th, random_state=40)
else:
    sampler = None

# Construir clasificador LGBM
lgbm = LGBMClassifier(random_state=314, metric='None', n_estimators=1000, verbosity=-1)

# Ensamblar pipeline con imblearn.Pipeline (soporta samplers en medio del pipeline)
if sampler is not None:
    lgbm_pipeline = ImbPipeline([('preprocessor', preprocessor), ('sampler', sampler), ('lgbm', lgbm)])
else:
    lgbm_pipeline = ImbPipeline([('preprocessor', preprocessor), ('lgbm', lgbm)])

# Hiperparámetros para RandomizedSearchCV (con prefijo lgbm__)
param_test = {
    'lgbm__num_leaves': sp_randint(6, 50),
    'lgbm__max_bin': sp_randint(60, 255),
    'lgbm__max_depth': sp_randint(5, 20),
    'lgbm__min_child_samples': sp_randint(100, 500),
    'lgbm__min_child_weight': [1e-5, 1e-3, 1e-2, 1e-1, 1, 1e1, 1e2, 1e3, 1e4],
    'lgbm__subsample': sp_uniform(loc=0.2, scale=0.8),
    'lgbm__colsample_bytree': sp_uniform(loc=0.4, scale=0.6),
    'lgbm__reg_alpha': [0, 1e-1, 1, 2, 5, 7, 10, 50, 100],
    'lgbm__reg_lambda': [0, 1e-5, 1e-3, 1e-2, 1e-1, 1, 5, 10, 20, 50, 100],
    'lgbm__scale_pos_weight': [1, 5, 20, 100],
    'lgbm__learning_rate': sp_uniform(loc=0.01, scale=0.1),
    'lgbm__subsample_freq': sp_randint(5, 20),
}

# Configurar RandomizedSearchCV (sin eval_set para no filtrar val en el CV)
random_search = RandomizedSearchCV(
    estimator=lgbm_pipeline,
    param_distributions=param_test,
    cv=3,
    scoring='roc_auc',
    n_jobs=-1,
    n_iter=60,
    refit=True,
    random_state=314,
)
random_search.fit(df_train[cols_for_model], y_train)

best_params = random_search.best_params_
print(f"Best AUC (CV): {random_search.best_score_:.3f}")
print(f"Best params: {best_params}")

# Re-entrenar con los mejores hiperparámetros y eval_set para early stopping.
# El eval_set debe estar PREPROCESADO porque el pipeline de sklearn/imblearn no
# transforma el eval_set automáticamente — solo transforma X_train internamente.
lgbm_pipeline.set_params(**best_params)
lgbm_pipeline.fit(df_train[cols_for_model], y_train)
X_val_transformed = lgbm_pipeline.named_steps['preprocessor'].transform(df_val[cols_for_model])
lgbm_pipeline.named_steps['lgbm'].set_params(
    **{
        k.replace('lgbm__', ''): v
        for k, v in best_params.items()
    }
)
lgbm_pipeline.fit(
    df_train[cols_for_model],
    y_train,
    lgbm__eval_metric=['auc'],
    lgbm__eval_set=[(X_val_transformed, y_val)],
    lgbm__eval_names=['valid'],
    lgbm__callbacks=[
        early_stopping(stopping_rounds=30, first_metric_only=True),
        log_evaluation(0),
    ],
)

# Predecir en test
y_pred_test_lgbm = lgbm_pipeline.predict_proba(df_test[cols_for_model])[:, 1]
resultado_final[f'{param_imb_method}-lgbm'] = y_pred_test_lgbm

In [ ]:
print("AUC Test:  %.3f" % roc_auc_score(y_test, y_pred_test_lgbm))

### Etapa 3: Evaluación de Modelos

El rendimiento de los modelos se evalúa mediante la métrica **AUC-ROC** (*Area Under the ROC Curve*).

La curva ROC es una curva de probabilidad: el AUC representa el grado de separabilidad entre clases. Indica en qué medida el modelo es capaz de distinguir entre usuarios fraudulentos y no fraudulentos. Cuanto más alto el AUC, mejor.

- **TPR (True Positive Rate)** = TP / (TP + FN) — tasa de verdaderos positivos
- **FPR (False Positive Rate)** = FP / (FP + TN) — tasa de falsos positivos

**Resultado del modelo LGBM en test:** AUC = 0.764
